In [1]:
import json
from pathlib import Path
from typing import List, Dict, Set, Tuple
import networkx as nx
import sys
import os
from pprint import pprint
import re

os.environ.pop("http_proxy", None)
os.environ.pop("https_proxy", None)
os.environ.pop("HTTP_PROXY", None)
os.environ.pop("HTTPS_PROXY", None)

# --- Config ---
PROJECT_ROOT = Path('../..').resolve()  # Adjust as needed to find the project root
GRAPH_PATH = PROJECT_ROOT / ".ai/context/internal/code_graph.gml"
DATABASE_PATH = PROJECT_ROOT / ".ai/context/internal/code_graph.json"
PRIORITY_PATH = PROJECT_ROOT / ".ai/context/internal/forward_pass_schedule.json"
DOX_DIR = PROJECT_ROOT / ".ai/context/internal/generated_dox"
XML_DIR = PROJECT_ROOT / ".ai/context/doxygen_output/xml"

OPENAI_MODEL = "gpt-4.1-nano"
OPENAI_MAX_TOKENS = None


# Add the modules directory to the Python path if needed
modules_dir = PROJECT_ROOT / '.ai/modules'
if str(modules_dir) not in sys.path:
    sys.path.append(str(modules_dir))

SRC_DIR = PROJECT_ROOT / 'src'
TEMP_DIR = PROJECT_ROOT / 'tmp'

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/qte2333/repos/legacy


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from llm_utils import call_openai
from gen_common import get_dependency_docs

In [4]:
import doxygen_parse as dp

db = dp.load_db(DATABASE_PATH)

In [5]:
import doxygen_graph as dg

# --- Load graph and priority ---
graph: nx.DiGraph = dg.load_graph(GRAPH_PATH)

2025-06-19 16:07:29.654 | INFO     | doxygen_graph:load_graph:450 - Graph loaded from /Users/qte2333/repos/legacy/.ai/context/internal/code_graph.gml, nodes: 14339, edges: 42761


In [6]:
import doc_db

In [7]:
# clear usages from all docs
if False:
    for cid, cid_docs in doc_db.docs.items():
        for sig, doc in cid_docs.items():
            doc.usages = None
        doc_db.save_docs(cid)

In [ ]:
if False:
    valid_keys = set()
    usage_dict = {}
    for cid, cid_docs in doc_db.docs.items():
        for sig, doc in cid_docs.items():
            valid_keys.add(f"{cid}, {sig}")
    for cid, cid_docs in doc_db.docs.items():
        changed = False
        for sig, doc in cid_docs.items():
            if doc.usages:
                for key in doc.usages:
                    cid, sig = key.split(", ", 1)
                    usage_dict.setdefault(cid, set()).add(sig)
                new_usages = {k: v for k, v in doc.usages.items() if k in valid_keys}
                if len(new_usages) < len(doc.usages):
                    print(f"*** Pruned usages for {doc.id}:")
                    for k, v in doc.usages.items():
                        if k not in new_usages:
                            print(f"  - {k}: {v}")
                    # doc.usages = new_usages
                    # changed = True
        if changed:
            doc_db.save_docs(cid)
    
    for cid, sigs in usage_dict.items():
        for sig in sigs:
            usage_dict.setdefault(cid, set()).add(sig)
            doc_db.docs[cid][sig].state = doc_db.DocumentState.GENERATED_USAGE
        doc_db.save_docs(cid)


In [8]:
def extract_function_name(name_str: str) -> str:
    """
    Extract the actual function name from various formats the LLM might return.
    
    Handles:
    - Return types: "String interpret_color_code" -> "interpret_color_code"
    - Parameters: "interpret_color_code(int arg)" -> "interpret_color_code"
    - Qualified names: "ClassName::method" -> "ClassName::method"
    
    Args:
        name_str: The function name string to process
        
    Returns:
        The cleaned function name
    """
    # Remove any parameter list
    if '(' in name_str:
        name_str = name_str.split('(', 1)[0].strip()
    
    # Handle return types by finding the last word
    # But preserve qualified names like Class::method
    parts = name_str.split()
    if len(parts) > 1:
        # Check if we have a qualified name with :: in the last part
        if '::' in parts[-1]:
            return parts[-1].strip()
        
        # Check if the qualified name spans multiple parts
        # e.g., "String Namespace::Class::method"
        for i in range(len(parts)-1, 0, -1):
            if '::' in parts[i]:
                return ' '.join(parts[i:]).strip()
        
        # If we got here, it's likely a return type followed by function name
        return parts[-1].strip()
    
    return name_str.strip()

# Test the function with various examples
test_cases = [
    "interpret_color_code",
    "String interpret_color_code",
    "interpret_color_code(int arg)",
    "String interpret_color_code(int arg)",
    "Class::method",
    "void Class::method",
    "void Class::method(int arg)",
    "Namespace::Class::method",
    "String Namespace::Class::method",
    "String Namespace::Class::method(int arg)"
]

for test in test_cases:
    print(f"{test} -> {extract_function_name(test)}")

interpret_color_code -> interpret_color_code
String interpret_color_code -> interpret_color_code
interpret_color_code(int arg) -> interpret_color_code
String interpret_color_code(int arg) -> interpret_color_code
Class::method -> Class::method
void Class::method -> Class::method
void Class::method(int arg) -> Class::method
Namespace::Class::method -> Namespace::Class::method
String Namespace::Class::method -> Namespace::Class::method
String Namespace::Class::method(int arg) -> Namespace::Class::method


In [ ]:
def get_dependency_docs(node_id: str) -> Dict[str, doc_db.Document]:
    deps = {}
    for dep_id in dg.fan_in(graph, node_id):
        dep_entity = db.get(dg.get_body_eid(db, dep_id))
        deps[dep_id] = doc_db.get_doc(dep_entity.id.compound, dep_entity.signature)

    return deps

def get_usage_summaries(node_id: str) -> Dict[str, str]:
    # get the entity object for the given node_id
    eid = dg.get_body_eid(db, node_id)
    entity = db.get(eid)
    usage_key = f"{entity.id.compound}, {entity.signature}"
    # print(f"usage key for {node_id} is: {usage_key}")

    # create a list of reference signatures for each dependency
    usage_docs = {}
    for dep_node_id in dg.fan_in(graph, node_id):
        dep_eid = dg.get_body_eid(db, dep_node_id)
        dep_entity = db.get(dep_eid)
        dep_doc = doc_db.get_doc(dep_entity.id.compound, dep_entity.signature)

        if dep_doc.usages and usage_key in dep_doc.usages:
            usage_docs[dep_node_id] = dep_doc.usages[usage_key]

    return usage_docs

def update_usage_summaries(node_id: str, usage_info: Dict[str, str]):
    eid = dg.get_body_eid(db, node_id)
    entity = db.get(eid)
    usage_key = f"{entity.id.compound}, {entity.signature}"
    # print(f"* insertion usage key for {node_id} is: {usage_key}")
    doc = doc_db.get_doc(entity.id.compound, entity.signature)
    doc.state = doc_db.DocumentState.GENERATED_USAGE
    doc_db.save_docs(entity.id.compound)

    for dep_id, usage in usage_info.items():
        if not usage:
            continue

        # print(f"Usage for {dep_id}: {usage}")
        dep_eid = dg.get_body_eid(db, dep_id)
        dep_entity = db.get(dep_eid)
        dep_doc = doc_db.get_doc(dep_entity.id.compound, dep_entity.signature)
        # dep_usages = {k:v for k,v in (dep_doc.usages or {}).items()}
        # dep_usages[usage_key] = usage
        # print(f"Updating usages for {dep_id} with dict items:")
        # for k,v in dep_usages.items():
        #     print(f"  {k}: {v}")
        if not dep_doc.usages:
            dep_doc.usages = {}
        dep_doc.usages[usage_key] = usage
        doc_db.save_docs(dep_entity.id.compound)

def create_usage_prompt(entity: dp.DoxygenEntity, code: str, dep_docs_to_complete: Dict[str, doc_db.Document]) -> str:
    # Skip if no dependencies with names to analyze
    if not dep_docs_to_complete:
        return None
    
    # create a prompt for the LLM
    prompt = f"""
You are analyzing how dependencies are used within a C++ function.

I will provide you with:
1. The full source code of a function
2. A list of its dependencies (functions, variables, constants, classes, macros, etc.)

Your task is to generate a JSON dictionary where:
- Each key is the ID of a dependency (as listed)
- Each value is a brief but specific explanation of how that dependency is used in this exact function

Focus your description on how the dependency appears in the logic — such as:
- whether it's used in a conditional or loop
- whether it determines control flow or permission
- whether it's called, checked, or updated
- whether it triggers side effects or returns values

Do not summarize what the dependency does in general — only how this function uses it.

Dependencies:
"""

    for name, doc in dep_docs_to_complete.items():
        kind: str = doc.kind
        if kind == "define":
            kind = "macro"
        if kind == "variable" and doc.cid.startswith('group_'):
            kind = "constant"
        prompt += f"- [{kind}] {name}"
        prompt += f" : {doc.brief}\n" if doc.brief else '\n'

    prompt += f"""

Source code:
```
{code.rstrip()}
```

Respond ONLY with a JSON dictionary. Each key should match a dependency name exactly. Each value must be one sentence describing how it is used.

Example:
{{
  "GET_RANK": "Used in permission checks to compare the character's level with visibility thresholds for different flag types.",
  "fsearch_obj": "Called at the end of the function to perform the object flag search after all validations have passed."
}}
"""

    # example = f"{{\n  {',\n  '.join(f'\"{s['name']}\": \"...\"' for s in list(dep_summaries.values())[:2])}\n}}"
    # prompt += example
    return prompt

def generate_usage_info(prompt: str):
    # --- Call OpenAI GPT-4o model ---
    response = call_openai(prompt, OPENAI_MODEL, OPENAI_MAX_TOKENS)

    if False:  # Set to True to debug response issues
        print("Raw response:")
        print(response)

    # --- Parse and structure the response ---
    try:
        # Remove any leading/trailing whitespace or newlines
        response = response.strip()

        # Handle empty responses
        if not response or response == "null":
            raise ValueError("Failed to generate usage information")

        # Attempt to parse the response as JSON
        usage_info = json.loads(response)

        print("\n*** loaded json response:")
        print(usage_info)
        return usage_info

    except json.JSONDecodeError as e:
        print(f"\n*** Error decoding JSON response: {e}")
        print(f"Response text: '{response}'")

        if not (response.startswith('{') and response.endswith('}')):
            raise

        response = response.strip("}{ \t\n\r")
        usage_info = {}

        # try to parse this by line
        for line in response.splitlines():
            key, value = line.split(": ", 1)
            key = key.strip().strip("\"")
            value = value.strip().strip("\"")
            usage_info[key] = value
            print(f"Parsed error usage for {key}: {value}")
        return usage_info

    return None

def analyze_dependency_usage(node_id: str):
    # get the source code for the node
    code_data = dg.get_code_body(PROJECT_ROOT, db, graph, node_id)
    if not code_data:
        return {}  # No body code to analyze
    body = code_data['code']

    eid = dg.get_body_eid(db, node_id)
    entity = db.get(eid)

    # if entity.name != 'do_storage':
    #     return

    dep_usages = get_usage_summaries(node_id)
    # print("*** Found usages:")
    # for dep_id, summary in dep_usages.items():
    #     print(f"  {dep_id} - {summary}")

    dep_docs = get_dependency_docs(node_id)
    # print("*** found dependency docs:")
    # print(sorted(dep_docs.keys()))

    name_id_map = {}
    incomplete_dep_docs = {}

    for dep_id, doc in dep_docs.items():
        # already documented?
        if dep_id in dep_usages:
            continue

        name = doc.name
        if doc.kind in ('function', 'variable', 'enum'):
            if any(doc.cid.startswith(k) for k in ('class', 'struct', 'namespace')):
                compound = db.get(doc.cid)
                name = f"{compound.name}::{doc.name}"

        # print(f'adding incomplete dep doc: {dep_id}, {name}')
        incomplete_dep_docs[name] = doc
        name_id_map[name] = dep_id
    # from pprint import pprint
    # pprint(name_id_map)

    # filter summaries to only include usages that are incomplete
    # incomplete_dep_docs = {k: v for k, v in dep_docs.items() if k not in dep_usages}
    if not incomplete_dep_docs:
        # print('skipping')
        return
    print("*** Found incomplete dependency docs:")
    print(sorted(incomplete_dep_docs.keys()))

    prompt = create_usage_prompt(entity, body, incomplete_dep_docs)
    # print('*** prompt')
    # print(prompt)

    name_usage_dict = generate_usage_info(prompt)
    # print('*** response')
    # pprint(name_usage_dict)

    # map names back to IDs
    dep_id_usage_dict = {}
    found_names = set()
    for oname, usage in name_usage_dict.items():
        # stupid chatgpt
        name = oname.strip()
        try:
            if name.startswith('['):
                name = name.split(']', 1)[1].strip()
            if any(name.startswith(k) for k in ('class', 'function', 'variable', 'constant', 'macro', 'struct', 'namespace', 'enum', 'define', 'typedef')):
                name = name.split(' ', 1)[1].strip()
            
            # Extract clean function name using our helper function
            name = extract_function_name(name)
            
            # Use regex only for final validation/extraction if needed
            name = re.match(r'[A-Za-z_][A-Za-z0-9_:]*', name).group(0)
        except:
            print(f"*** unable to parse name: '{name}' (from '{oname}')")
        if name not in name_id_map:
            print(f"*** skipping unknown name: '{name}' (from '{oname}')")
            continue
        if name not in incomplete_dep_docs:
            continue
        print(f"*** updating {name} ({name_id_map.get(name, name)}) with: {usage}")
        found_names.add(name)
        dep_id_usage_dict[name_id_map.get(name, name)] = usage
# print(name_id_map[name], usage)

    update_usage_summaries(node_id, dep_id_usage_dict)

    for name in incomplete_dep_docs:
        if name not in found_names:
            print(f"*** {name} is unsatisfied")
            print(set(name_usage_dict.keys()))

# Example of using the dependency analysis function
# Choose a function or class with dependencies
if False:
    sample_node_id = '1a447f1b6b8a0dc34de8122b00748531b3'  # Use a real ID from your graph
    usage_info = analyze_dependency_usage(sample_node_id)
else:
    with PRIORITY_PATH.open('r') as f:
        priority_order: List[Dict[str, str]] = json.load(f)
    count = 0
    for item in reversed(priority_order):
        count += 1
        # if count < 4: continue
        # if count > 4: break
        usage_info = analyze_dependency_usage(item['id'])
        print(f"{count}/{len(priority_order):>4}: {item['id']} - {item['name']} - {item['kind']}")


1/1963: 1ad30707c36be51b3847ca582f8ea2ec73 - void do_enter(Character *ch, String argument) - function
2/1963: 1adc2cda21737d2fbf99a48087a6cfdd4f - void do_create(Character *ch, String argument) - function
3/1963: 1ac10cd19233ef8720d6253d6a6d0c5ac2 - void do_tail(Character *ch, String argument) - function
4/1963: 1a2f2a1bb67cd93bd35e4a80dacc76d27c - void do_warp(Character *ch, String argument) - function
5/1963: 1aa5343b3d3064050785d33781e0b8ad3a - void do_wizgroup(Character *ch, String argument) - function
6/1963: 1a28fa6df8ca72610d8f661cc231c73c30 - void worldmap::Quadtree< T >::remove(const Coordinate &coord) - function
7/1963: 1a44722339106ecf7c5e37ace531a971a7 - void do_printhelps(Character *ch, String argument) - function
8/1963: 1aced03971d13897c52daaf024b0efebc1 - void do_storage(Character *ch, String argument) - function
9/1963: 1a1f7f87f2f8bb4d280c5525257514aa5b - void recall(Character *ch, bool clan) - function
10/1963: 1a3557fe3aac6f7de14f9f49ca9c8088fe - void do_clan_recall